In [1]:
# pip install matplotlib

In [2]:
from importlib.metadata import version
print("torch verison ", version("torch"))
print("tiktoken version ", version("tiktoken"))

torch verison  2.8.0
tiktoken version  0.13.0


In [6]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "data/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)

In [7]:
with open("the-verdict.txt", "r", encoding="UTF-8") as f:
    raw_text = f.read()
print("Lenght of text: ", len(raw_text))
print(raw_text[:20])

Lenght of text:  20479
I HAD always thought


In [16]:
import re
text = "Hello, world! Ross: How you doing?"
vocab = re.split(r'\s', text)
print(vocab)
print(len(vocab))
vocab = re.split(r'(\s)', text)
print(vocab)
print(len(vocab))
vocab = re.split(r'([,.:;?!_"()\']|--|\s)', text)
print(vocab)
print(len(vocab))
vocab = [item.strip() for item in vocab if item.strip()]
print(vocab)
print(len(vocab))

['Hello,', 'world!', 'Ross:', 'How', 'you', 'doing?']
6
['Hello,', ' ', 'world!', ' ', 'Ross:', ' ', 'How', ' ', 'you', ' ', 'doing?']
11
['Hello', ',', '', ' ', 'world', '!', '', ' ', 'Ross', ':', '', ' ', 'How', ' ', 'you', ' ', 'doing', '?', '']
19
['Hello', ',', 'world', '!', 'Ross', ':', 'How', 'you', 'doing', '?']
10


In [22]:
preprocessed = re.split(r'([,._?!:;"\'()]|--|\s)', raw_text)
print("Length: ", len(preprocessed))
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print("Length: ", len(preprocessed))
print(preprocessed[:20])

Length:  9235
Length:  4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']


vocab to token id

In [24]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [26]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >=10:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)


In [39]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        print(type(vocab))
        self.int_to_str = {i:s for s, i in vocab.items()}
    def encode(self, text):
        preprocessed = re.split(r'([,.:;"!?()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [40]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
        Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

<class 'dict'>
[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [34]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [35]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [41]:
text = "Hello, do you like tea. Is this-- a test?"
tokenizer.encode(text)

KeyError: 'Hello'

In [43]:
print(len(set(preprocessed)))
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
print(len(all_tokens))

1130
1132


In [49]:
vocab = {token : integer for integer, token in enumerate(all_tokens)}
len(vocab.items())


1132

In [50]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [51]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:tok for tok, i in vocab.items()}

    def encode(self, text):
        processed = re.split(r'([,.:;?!"()\']|--|\s)', text)
        processed = [item.strip() for item in processed if item.strip()]
        processed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in processed
        ]
        ids = [self.str_to_int[s] for s in processed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [54]:
tokenizer = SimpleTokenizerV2(vocab)
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " <|endoftext|> ".join((text1, text2))
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [55]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [56]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

In [57]:
hudai = "Abc, dhur--  ?check , "
text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', hudai)
print(text)

Abc, dhur--?check, 


Byte-Pair Encoding

In [58]:
import importlib
import tiktoken

print("tiktoken version: ", importlib.metadata.version("tiktoken"))

tiktoken version:  0.13.0


In [59]:
tokenizer = tiktoken.get_encoding("gpt2")

In [60]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
    "of someunknownPlace."
)
ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(ids)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [61]:
strings = tokenizer.decode(ids)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


In [63]:
with open("the-verdict.txt", "r", encoding="UTF-8")as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(enc_text[-5:])
print("length: ", len(enc_text))

[674, 1611, 286, 1242, 526]
length:  5145


In [64]:
enc_sample = enc_text[50:]

In [66]:
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [68]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "-->", desired)

[290] --> 4920
[290, 4920] --> 2241
[290, 4920, 2241] --> 287
[290, 4920, 2241, 287] --> 257


In [71]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    target = enc_sample[i]
    print(tokenizer.decode(context), "--->", tokenizer.decode([target]))

 and --->  established
 and established --->  himself
 and established himself --->  in
 and established himself in --->  a
